# Spark SQL & UDFs

## What's covered

- `spark.sql(...)` and the same-Catalyst-plan guarantee (SQL ≡ DataFrame)
- Temporary views: session, global, and persisted catalog tables
- Basic SQL: `SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, `LIMIT`
- Null semantics — why `col == null` always returns null, and the `<=>` operator
- CTEs (`WITH ... AS`)
- All six SQL `JOIN` types
- Subqueries — scalar, `IN`, correlated `EXISTS`
- Window functions in SQL syntax
- `monotonically_increasing_id` — what it actually guarantees (and what it doesn't)
- The `spark.catalog` API
- Numeric, advanced-date, and higher-order array functions
- Python UDFs vs Pandas UDFs vs `applyInPandas` — and why the planner goes blind through a UDF
- The performance-tier decision table


## SQL is a first-class citizen

`spark.sql(...)` accepts any ANSI SQL string and routes it through the **same Catalyst optimizer** as the DataFrame API. SQL and DataFrame paths produce identical execution plans. Choose whichever form is more readable; mix them freely in the same application.

> **ANSI SQL mode** — Spark 3.0+ ships a stricter SQL semantics: integer overflow throws instead of silently wrapping, casts are stricter, divide-by-zero raises. Enable with `spark.conf.set("spark.sql.ansi.enabled", "true")`. Off by default for backward compatibility, but recommended for new pipelines.

![Spark SQL Catalyst](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-sql-catalyst.svg)


## Setup — register the seven bank tables as views

We register every bank table from `data/` as a session-scoped temp view. From here on, any `spark.sql("SELECT ... FROM <table>")` call works without re-loading.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType, DecimalType,
)

spark = (
    SparkSession.builder
    .appName("SparkSQLAndUDFs")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "data"

# Explicit schemas only for the CSVs — Parquet/ORC/JSON-with-schema cover themselves.
customers_schema = StructType([
    StructField("customer_id",   StringType(), nullable=False),
    StructField("full_name",     StringType()),
    StructField("email",         StringType()),
    StructField("phone",         StringType()),
    StructField("date_of_birth", DateType()),
    StructField("gender",        StringType()),
    StructField("city",          StringType()),
    StructField("state",         StringType()),
    StructField("country",       StringType()),
    StructField("created_at",    TimestampType()),
])

loan_repayments_schema = StructType([
    StructField("repayment_id", StringType(), nullable=False),
    StructField("loan_id",      StringType(), nullable=False),
    StructField("customer_id",  StringType(), nullable=False),
    StructField("due_date",     DateType()),
    StructField("paid_date",    DateType()),
    StructField("due_amount",   DecimalType(18, 2)),
    StructField("paid_amount",  DecimalType(18, 2)),
    StructField("status",       StringType()),
])

tables = {
    "customers":            spark.read.schema(customers_schema).option("header", "true").csv(f"{DATA_DIR}/customers"),
    "loan_repayments":      spark.read.schema(loan_repayments_schema).option("header", "true").csv(f"{DATA_DIR}/loan_repayments"),
    "card_accounts":        spark.read.json(f"{DATA_DIR}/card_accounts"),
    "loan_accounts":        spark.read.json(f"{DATA_DIR}/loan_accounts"),
    "card_transactions":    spark.read.parquet(f"{DATA_DIR}/card_transactions"),
    "bank_accounts":        spark.read.orc(f"{DATA_DIR}/bank_accounts"),
    "account_transactions": spark.read.orc(f"{DATA_DIR}/account_transactions"),
}

for name, df in tables.items():
    df.createOrReplaceTempView(name)
    print(f"  registered: {name:24s} ({df.count():>6} rows)")


## Temporary views — session, global, and managed tables

Three lifetimes, three call sites:

| Method | Lifetime | Address as |
|---|---|---|
| `df.createOrReplaceTempView(name)` | This SparkSession only | `<name>` |
| `df.createOrReplaceGlobalTempView(name)` | This JVM (all sessions) | `global_temp.<name>` |
| `df.write.saveAsTable(name)` | Persisted in the metastore (notebook 04) | `<name>` |

The **SQL ↔ DataFrame round-trip** pattern: register a DataFrame as a view, query it with SQL, take the result back as a DataFrame, continue chaining. Spark routes each hop through the same optimizer, so plans stay merged.


In [ ]:
# Global temp view — visible from any session in this JVM, addressed via global_temp.<name>
spark.sql("""
    SELECT customer_id, full_name, city
    FROM   customers
    WHERE  city = 'Mumbai'
""").createOrReplaceGlobalTempView("mumbai_customers")

spark.sql("SELECT COUNT(*) AS n FROM global_temp.mumbai_customers").show()

# SQL ↔ DataFrame round-trip — start in SQL, finish in DataFrame
from pyspark.sql.functions import col

sql_result = spark.sql("""
    SELECT customer_id, COUNT(*) AS n_txns
    FROM   card_transactions
    WHERE  status = 'APPROVED'
    GROUP BY customer_id
""")

(
    sql_result
    .filter(col("n_txns") > 5)
    .orderBy(col("n_txns").desc())
    .show(5)
)


## Basic SQL queries

`SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`, `LIMIT` — all standard ANSI. Spark's SQL **does** support `HAVING`, even though there is no `.having()` method on DataFrames.


In [ ]:
# Simple SELECT with WHERE and ORDER BY
spark.sql("""
    SELECT transaction_id, merchant_category, amount, status
    FROM   card_transactions
    WHERE  amount > 10000 AND status = 'APPROVED'
    ORDER BY amount DESC
    LIMIT 5
""").show(truncate=False)

# GROUP BY + HAVING — categories with more than 100 approved transactions
spark.sql("""
    SELECT merchant_category,
           COUNT(*)              AS n_txns,
           ROUND(SUM(amount), 2) AS total
    FROM   card_transactions
    WHERE  status = 'APPROVED'
    GROUP BY merchant_category
    HAVING COUNT(*) > 100
    ORDER BY total DESC
""").show(truncate=False)


## Null semantics — `col == null` always returns null

Three-valued logic from ANSI SQL applies, with one DataFrame-specific trap:

| Expression | Result |
|---|---|
| `col == null` (Python `==` against `None`) | **Always `null`**, never `true` — therefore never matches in a filter |
| `col.isNull()` / `col.isNotNull()` | The DataFrame way to test for null |
| SQL `col IS NULL` / `col IS NOT NULL` | The SQL way |
| `col <=> null` / `col.eqNullSafe(null)` | **Null-safe equality** — returns `true` when both sides are null |
| `col == col2` when either side is null | Returns `null` (rows filtered out unless using `eqNullSafe`) |

The trap is the first row. Writing `df.filter(col("merchant_category") == None)` returns **zero rows**, because `category == null` evaluates to `null` for every row, and a filter passes only rows where the predicate is `true`. The fix is `col(...).isNull()`.

Same trap in SQL: `WHERE col = NULL` never matches. Use `WHERE col IS NULL`.


In [ ]:
from pyspark.sql.functions import lit, when

# Build a small frame with a deliberately null row
df_null = spark.createDataFrame(
    [(1, "Travel"), (2, None), (3, "Food")],
    ["id", "category"],
)

# WRONG — the "obvious" Python equality returns zero rows
print("--- col == None  (WRONG) ---")
df_null.filter(col("category") == None).show()                   # 0 rows

# RIGHT — isNull / isNotNull
print("--- col.isNull()  (RIGHT) ---")
df_null.filter(col("category").isNull()).show()                  # 1 row

# In SQL
spark.sql("""
    SELECT id, category, category = NULL AS eq_null, category IS NULL AS is_null
    FROM   VALUES (1, 'Travel'), (2, NULL), (3, 'Food') AS t(id, category)
""").show()

# Null-safe equality with <=> / eqNullSafe — null matches null
print("--- eqNullSafe ---")
df_null.withColumn("matches_null", col("category").eqNullSafe(lit(None))).show()


## CTEs — `WITH` clause

Common Table Expressions name intermediate results so a complex query reads top-to-bottom instead of inside-out. Spark supports full ANSI `WITH ... AS (...)` chains.


In [ ]:
# Top spender per city — chained CTEs + a final ROW_NUMBER pass
spark.sql("""
    WITH approved_spend AS (
        SELECT customer_id, SUM(amount) AS total_spend
        FROM   card_transactions
        WHERE  status = 'APPROVED'
        GROUP BY customer_id
    ),
    customer_with_spend AS (
        SELECT c.city, c.full_name, s.total_spend
        FROM   customers c
        JOIN   approved_spend s USING (customer_id)
    )
    SELECT city,
           full_name,
           ROUND(total_spend, 2) AS total_spend
    FROM (
        SELECT city, full_name, total_spend,
               ROW_NUMBER() OVER (PARTITION BY city ORDER BY total_spend DESC) AS rn
        FROM   customer_with_spend
    )
    WHERE rn = 1
    ORDER BY total_spend DESC
    LIMIT 5
""").show(truncate=False)


## SQL JOINs

All six SQL join types Spark supports — same semantics as the DataFrame `how` argument from notebook 05, just SQL syntax.

| Join type | Returns |
|---|---|
| `INNER JOIN` | Rows matching in both tables |
| `LEFT JOIN` | All left rows; nulls for unmatched right |
| `RIGHT JOIN` | All right rows; nulls for unmatched left |
| `FULL OUTER JOIN` | All rows from both; nulls where no match |
| `LEFT SEMI JOIN` | Left rows that have a match in right (no right columns) |
| `LEFT ANTI JOIN` | Left rows that have **no** match in right |


In [ ]:
# Customer-360 — three tables joined in one SQL query
spark.sql("""
    SELECT c.customer_id,
           c.full_name,
           c.city,
           COUNT(DISTINCT t.transaction_id)  AS n_card_txns,
           ROUND(SUM(t.amount), 2)           AS total_card_spend,
           COUNT(DISTINCT l.loan_id)         AS n_loans,
           ROUND(SUM(l.principal_amount), 2) AS total_loan_principal
    FROM   customers c
    LEFT JOIN card_transactions t
           ON c.customer_id = t.customer_id AND t.status = 'APPROVED'
    LEFT JOIN loan_accounts l
           ON c.customer_id = l.customer_id
    GROUP BY c.customer_id, c.full_name, c.city
    ORDER BY total_card_spend DESC NULLS LAST
    LIMIT 10
""").show(truncate=False)


## Subqueries

Spark SQL supports correlated and uncorrelated subqueries in `WHERE`, `FROM`, and `SELECT`.

- **Scalar subquery** — single value in `SELECT` or `WHERE`.
- **`IN` subquery** — set membership.
- **`EXISTS` subquery** — correlated; outer row passes if the inner query has any matching row.


In [ ]:
# Scalar subquery — transactions above the overall average
spark.sql("""
    SELECT transaction_id, amount
    FROM   card_transactions
    WHERE  amount > (
        SELECT AVG(amount) FROM card_transactions WHERE status = 'APPROVED'
    )
    LIMIT 5
""").show()

# IN subquery — transactions from customers who have an active loan
spark.sql("""
    SELECT *
    FROM   account_transactions
    WHERE  customer_id IN (
        SELECT customer_id FROM loan_accounts WHERE status = 'ACTIVE'
    )
    LIMIT 5
""").show(truncate=False)

# EXISTS subquery — customers who have at least one DECLINED transaction
spark.sql("""
    SELECT c.customer_id, c.full_name
    FROM   customers c
    WHERE  EXISTS (
        SELECT 1
        FROM   card_transactions t
        WHERE  t.customer_id = c.customer_id AND t.status = 'DECLINED'
    )
    LIMIT 5
""").show(truncate=False)


## Window functions in SQL

The DataFrame Window API from notebook 05, in SQL syntax: `OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)`. Same semantics — `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `NTILE(n)`, `LAG(col, n)`, `LEAD(col, n)`, plus running aggregates over a frame.


In [ ]:
# Per-customer transaction rank, plus a running cumulative spend
spark.sql("""
    SELECT customer_id,
           transaction_id,
           amount,
           transaction_at,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id
               ORDER BY amount DESC
           ) AS amount_rank,
           ROUND(SUM(amount) OVER (
               PARTITION BY customer_id
               ORDER BY transaction_at
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ), 2) AS cum_spend
    FROM   card_transactions
    WHERE  status = 'APPROVED'
    ORDER BY customer_id, transaction_at
    LIMIT 10
""").show(truncate=False)


## `monotonically_increasing_id` — monotonic, not sequential

People reach for `monotonically_increasing_id()` looking for a row number. **It is not one.** The contract is the literal name:

- **Monotonic** — within a single execution, every ID is strictly greater than every previous ID generated in the same partition.
- **Not sequential** — the function emits 63-bit IDs whose upper 31 bits encode the partition index and lower 33 bits a per-partition counter. Across partitions you get big jumps (e.g., `0, 1, 2, ..., 8589934592, 8589934593, ...`).
- **Not deterministic across runs** — a re-execution can assign different IDs to the same row, because partition assignment can shift.

When you want **dense, sequential, deterministic** numbering, use `row_number()` over a window — that's what it's for:

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
df.withColumn("rn", row_number().over(Window.orderBy("created_at")))
```

When you want a **cheap unique ID** and don't care about gaps, `monotonically_increasing_id()` is fine.

When you want a **stable surrogate key across runs**, neither works — generate UUIDs (`expr("uuid()")`) or derive a hash of business-key columns (`sha2(concat_ws(...), 256)`).


In [ ]:
from pyspark.sql.functions import monotonically_increasing_id, row_number
from pyspark.sql.window import Window

# Multi-partition input so the jump is visible
df_multi = spark.range(10).repartition(3)

# monotonically_increasing_id — gaps at partition boundaries
df_multi.withColumn("mid", monotonically_increasing_id()).orderBy("mid").show()

# row_number — dense and sequential, at the cost of a global shuffle
df_multi.withColumn(
    "rn", row_number().over(Window.orderBy("id"))
).orderBy("rn").show()


## The catalog API

`spark.catalog` lets you list and inspect databases, tables, views, and columns programmatically — useful for discovery and for cleanup.

| Call | Returns |
|---|---|
| `spark.catalog.listDatabases()` | All databases |
| `spark.catalog.listTables()` | Tables and views in the current database |
| `spark.catalog.listColumns(table)` | Columns of a table |
| `spark.catalog.tableExists(name)` | Boolean |
| `spark.catalog.dropTempView(name)` | Drop a session view |
| `spark.catalog.dropGlobalTempView(name)` | Drop a global view |


In [ ]:
print("databases :", [db.name for db in spark.catalog.listDatabases()])
print("views     :", sorted([t.name for t in spark.catalog.listTables() if t.isTemporary]))
print("customers exists?", spark.catalog.tableExists("customers"))

# Cleanup the global view we created earlier
spark.catalog.dropGlobalTempView("mumbai_customers")
print("global cleanup done")


## DataFrame API vs SQL — same plan

The same query, written both ways. Catalyst rewrites both into the same physical plan — the proof is in `.explain()`.


In [ ]:
from pyspark.sql.functions import sum as _sum, count, round as _round

# DataFrame chain
df_query = (
    spark.table("card_transactions")
    .filter(col("status") == "APPROVED")
    .groupBy("merchant_category")
    .agg(
        count("*").alias("n_txns"),
        _round(_sum("amount"), 2).alias("total"),
    )
    .orderBy(col("total").desc())
)

# Equivalent SQL
sql_query = spark.sql("""
    SELECT merchant_category,
           COUNT(*)              AS n_txns,
           ROUND(SUM(amount), 2) AS total
    FROM   card_transactions
    WHERE  status = 'APPROVED'
    GROUP BY merchant_category
    ORDER BY total DESC
""")

df_query.show(3)
sql_query.show(3)

print("--- DataFrame plan ---"); df_query.explain()
print("--- SQL plan ---");        sql_query.explain()


## Numeric functions

Beyond the obvious arithmetic, the function library has a handful of numeric helpers worth knowing for financial work.

| Function | Purpose |
|---|---|
| `abs(col)` | Absolute value |
| `ceil(col)`, `floor(col)` | Round up / down to nearest integer |
| `round(col, n)` | Round half up to n decimal places |
| `bround(col, n)` | Banker's rounding — round half to **even** |
| `pow(col, n)` | Raise to a power |
| `sqrt(col)`, `log(base, col)`, `exp(col)` | Exponential / log |
| `signum(col)` | -1, 0, or 1 by sign |
| `greatest(*cols)` | Row-wise max across multiple columns |
| `least(*cols)` | Row-wise min across multiple columns |


In [ ]:
from pyspark.sql.functions import (
    ceil, floor, bround, greatest, least,
)

(
    spark.table("card_transactions")
    .select(
        col("amount"),
        ceil(col("amount")).alias("ceil"),
        floor(col("amount")).alias("floor"),
        _round(col("amount"), 0).alias("round_half_up"),
        bround(col("amount"), 0).alias("round_half_even"),
    )
    .show(5)
)

# greatest / least — row-wise across columns. Useful for caps and floors.
(
    spark.table("loan_accounts")
    .select(
        "loan_id",
        "principal_amount",
        greatest(col("principal_amount"), lit(50000)).alias("floored_at_50k"),
        least(col("principal_amount"), lit(2000000)).alias("capped_at_2m"),
    )
    .show(5)
)


## Advanced date functions

Building on notebook 05's date basics, these handle reporting periods and calendar tricks.

| Function | Purpose |
|---|---|
| `date_trunc("hour"/"day"/"month", col)` | Truncate a timestamp to a unit |
| `last_day(col)` | Last day of the month containing `col` |
| `next_day(col, "MON")` | Next named weekday on or after `col` |
| `months_between(end, start)` | Months including a fractional part |
| `quarter(col)` | Calendar quarter (1–4) |
| `weekofyear(col)` | ISO week number |


In [ ]:
from pyspark.sql.functions import (
    date_trunc, last_day, months_between, quarter, weekofyear, current_date,
)

(
    spark.table("card_transactions")
    .select(
        col("transaction_at"),
        date_trunc("hour", col("transaction_at")).alias("hour_bucket"),
        last_day(col("transaction_at").cast("date")).alias("month_end"),
        quarter(col("transaction_at")).alias("calendar_qtr"),
        weekofyear(col("transaction_at")).alias("iso_week"),
    )
    .show(5, truncate=False)
)

# months_between — tenure elapsed for active loans
(
    spark.table("loan_accounts")
    .filter(col("status") == "ACTIVE")
    .select(
        "loan_id",
        "disbursed_at",
        months_between(current_date(), col("disbursed_at").cast("date")).alias("months_elapsed"),
    )
    .show(5)
)


## Higher-order array functions

Higher-order functions take a **lambda** as an argument and apply it to array elements *on the JVM* — no UDF serialization overhead. They turn many would-be UDFs into one-liners.

| Function | Purpose |
|---|---|
| `transform(arr, lambda)` | Map over array elements |
| `filter(arr, lambda)` | Keep elements where the lambda is true |
| `aggregate(arr, zero, merge)` | Fold an array to a scalar |
| `exists(arr, lambda)` | True if any element matches |
| `forall(arr, lambda)` | True if all elements match |
| `zip_with(a, b, lambda)` | Element-wise combine two arrays |


In [ ]:
from pyspark.sql.functions import (
    array, transform, filter as array_filter,
    aggregate, exists, forall, lower,
)

# A tiny one-row playground with two array columns to operate on
demo = spark.range(1).select(
    array(lit("HIGH_AMOUNT"), lit("FRAUD_FLAG"), lit("DECLINED_TXN")).alias("reasons"),
    array(lit(1200.0), lit(18500.0), lit(450.0), lit(6700.0)).alias("amounts"),
)

(
    demo.select(
        col("reasons"),
        # transform — lowercase every element
        transform(col("reasons"), lambda x: lower(x)).alias("reasons_lower"),
        # filter — keep only elements containing 'FRAUD'
        array_filter(col("reasons"), lambda x: x.contains("FRAUD")).alias("fraud_only"),
        # exists — does any reason contain 'FLAG'?
        exists(col("reasons"), lambda x: x.contains("FLAG")).alias("has_flag"),
        # forall — does every reason contain an underscore?
        forall(col("reasons"), lambda x: x.contains("_")).alias("all_underscored"),
    )
    .show(truncate=False)
)

# aggregate — sum of an array column without an explode + groupBy round-trip
(
    demo.select(
        col("amounts"),
        aggregate(col("amounts"), lit(0.0), lambda acc, x: acc + x).alias("total"),
    )
    .show(truncate=False)
)


## Python UDFs — `@udf` and `spark.udf.register`

When a built-in cannot express your logic, fall back to a Python UDF. Two registration paths:

- **`@udf(returnType)`** — decorator for the DataFrame API. Use the result as a regular column.
- **`spark.udf.register(name, func, returnType)`** — register a name; same UDF callable from `spark.sql(...)`.

The cost is real. Every row crosses the JVM↔Python boundary twice — once to send the value to a Python worker, once to bring the result back. **10–100× slower than equivalent built-ins** on large data.

**Key note — a Python UDF is a Catalyst black box.** Catalyst can't see inside it, which means:

- **No predicate pushdown across the UDF.** `df.filter(my_udf(col("x")) > 5)` cannot push that filter into the file scan — the entire column must be read and the UDF run on every row before the filter can be evaluated.
- **No column pruning across the UDF.** If the UDF takes a row-shaped object (e.g., a `struct` column), Spark reads every field of the struct even when the UDF uses only one.
- **No null safety.** A built-in like `upper(col)` returns `null` when given `null` (SQL-standard). A naive Python UDF gets called with `None` and may crash unless you guard it.
- **No type checking at plan time.** A return-type mismatch surfaces at runtime, not at job submission.

Always exhaust built-ins, higher-order functions, and Pandas UDFs first.

![UDF Performance](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-udf-performance.svg)


In [ ]:
from pyspark.sql.functions import udf

def mask_card(card_id: str) -> str:
    """Keep first 4 and last 4 characters of a card id; mask the middle."""
    if card_id is None or len(card_id) < 8:
        return card_id
    return card_id[:4] + ("*" * (len(card_id) - 8)) + card_id[-4:]

# DataFrame API — wrap as a UDF and call as a regular column function
mask_card_udf = udf(mask_card, returnType=StringType())
(
    spark.table("card_accounts")
    .select("card_id", mask_card_udf(col("card_id")).alias("masked"))
    .show(5)
)

# SQL API — register the same function under a name and call from SQL
spark.udf.register("mask_card", mask_card, returnType=StringType())
spark.sql("""
    SELECT card_id, mask_card(card_id) AS masked
    FROM   card_accounts
    LIMIT 5
""").show()


## Pandas UDFs — vectorized via Apache Arrow

Pandas UDFs (also called **vectorized UDFs**) transfer data in **columnar Arrow batches** rather than row-by-row. Inside the UDF, you operate on a `pd.Series` (or a `pd.DataFrame` for grouped variants). NumPy and pandas vectorization apply, and Arrow eliminates most of the serialization overhead.

Three flavors worth knowing:

| Flavor | Input | Output | When |
|---|---|---|---|
| **Series → Series** (default) | `pd.Series` per column | `pd.Series` | Most row-wise transformations |
| **Iterator of Series** | Iterator of `pd.Series` | Iterator of `pd.Series` | Setup once (load model, open DB) before processing batches |
| **`groupBy().applyInPandas(f, schema)`** | `pd.DataFrame` per group | `pd.DataFrame` | Custom aggregation needing full pandas |


In [ ]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

# Series -> Series — convert INR amounts to USD at a fixed rate (toy example)
@pandas_udf(DoubleType())
def inr_to_usd(amount: pd.Series) -> pd.Series:
    return amount.astype(float) / 83.0

(
    spark.table("card_transactions")
    .filter(col("status") == "APPROVED")
    .select(
        "transaction_id",
        col("amount").alias("amount_inr"),
        inr_to_usd(col("amount")).alias("amount_usd"),
    )
    .show(5)
)


In [ ]:
from pyspark.sql.types import LongType

# applyInPandas — custom group aggregation that needs full pandas (e.g., median)
agg_schema = StructType([
    StructField("customer_id",   StringType()),
    StructField("n_txns",        LongType()),
    StructField("total_amount",  DoubleType()),
    StructField("median_amount", DoubleType()),
])

def per_customer_stats(pdf: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([{
        "customer_id":   pdf["customer_id"].iloc[0],
        "n_txns":        len(pdf),
        "total_amount":  float(pdf["amount"].sum()),
        "median_amount": float(pdf["amount"].median()),
    }])

(
    spark.table("card_transactions")
    .filter(col("status") == "APPROVED")
    .groupBy("customer_id")
    .applyInPandas(per_customer_stats, schema=agg_schema)
    .orderBy(col("total_amount").desc())
    .show(5)
)


## Performance tiers — when to use each

| Scenario | Recommended approach | Why |
|---|---|---|
| String / date / array / conditional / numeric work | **Built-in `pyspark.sql.functions`** | Tungsten codegen, no serialization |
| Element-wise array/map work | **Higher-order functions** (`transform`, `filter`, `aggregate`) | Lambda runs on the JVM |
| NumPy/pandas/scikit-learn vectorized work | **Pandas UDF — Series → Series** | Arrow columnar batches |
| Setup-once patterns (load model, open DB connection) | **Pandas UDF — Iterator** | One setup per batch instead of per row |
| Custom group-level logic that needs full pandas | **`applyInPandas`** | One pandas DataFrame per group |
| Simple custom logic, small data, no built-in alternative | **Plain `@udf`** | Acceptable; avoid in hot paths |
| Plain `@udf` on billions of rows | **Avoid** — rewrite as built-ins or Pandas UDF | 10–100× slower, GC pressure |


In [ ]:
spark.stop()
